In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import email
from email import policy
from email.parser import Parser
import hashlib
import re

FILE_PATH = "drive/MyDrive/phishing project datasets/emails.csv"

pd.set_option('display.max_colwidth', 200)


In [ ]:
df_raw = pd.read_csv(FILE_PATH)

print(f"Loaded {len(df_raw):,} rows")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)


Loaded 517,401 rows
Columns: ['file', 'message']


,file,message
0,allen-p/_sent_mail/1.,"Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nConte..."
1,allen-p/_sent_mail/10.,"Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May 2001 13:51:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: john.lavorato@enron.com\nSubject: Re:\nMime-Version: 1.0\n..."
2,allen-p/_sent_mail/100.,"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme>\nDate: Wed, 18 Oct 2000 03:00:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: leah.arsdall@enron.com\nSubject: Re: test\nMime-Version: ..."


In [ ]:
assert 'file' in df_raw.columns, "Expected a 'file' column (Enron folder path) — check the CSV format."
assert 'message' in df_raw.columns, "Expected a 'message' column (raw email text) — check the CSV format."
assert len(df_raw) > 0, "Dataframe is empty — check FILE_PATH."

print("PASS: columns and row count look correct.")
print()
print("Sample raw 'message' field (first 500 chars):")
print("-" * 60)
print(df_raw.loc[0, 'message'][:500])

PASS: columns and row count look correct.

Sample raw 'message' field (first 500 chars):
------------------------------------------------------------
Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 


In [ ]:
n_before = len(df_raw)

# Drop rows that are exact duplicates on the raw message text
df_dedup = df_raw.drop_duplicates(subset=['message'], keep='first').copy()

# Defensive: drop duplicate file paths too, if any exist
df_dedup = df_dedup.drop_duplicates(subset=['file'], keep='first').copy()

n_after = len(df_dedup)
n_removed = n_before - n_after

print(f"Rows before dedup: {n_before:,}")
print(f"Rows after dedup:  {n_after:,}")
print(f"Exact duplicates removed: {n_removed:,} ({n_removed / n_before:.2%})")

df_dedup = df_dedup.reset_index(drop=True)


Rows before dedup: 517,401
Rows after dedup:  517,401
Exact duplicates removed: 0 (0.00%)


In [ ]:
dup_msg_count = df_dedup.duplicated(subset=['message']).sum()
dup_file_count = df_dedup.duplicated(subset=['file']).sum()

assert dup_msg_count == 0, f"Still {dup_msg_count} duplicate messages remaining!"
assert dup_file_count == 0, f"Still {dup_file_count} duplicate file paths remaining!"
assert len(df_dedup) <= n_before, "Dedup should never increase row count."

print("PASS: no exact duplicate messages or file paths remain.")
print(f"Working row count: {len(df_dedup):,}")

PASS: no exact duplicate messages or file paths remain.
Working row count: 517,401


In [ ]:
def parse_email_message(raw_text):
    """Parse a raw RFC-822 email string into a dict of headers + body.

    Never raises — malformed input is captured via parse_success/parse_error
    so it can be handled explicitly in the next pipeline stage.
    """
    result = {
        'message_id': None, 'date': None, 'from': None, 'to': None,
        'subject': None, 'cc': None, 'bcc': None,
        'x_from': None, 'x_to': None, 'x_folder': None,
        'x_origin': None, 'x_filename': None, 'content_type': None,
        'body': None, 'parse_success': False, 'parse_error': None,
    }

    if not isinstance(raw_text, str) or raw_text.strip() == '':
        result['parse_error'] = 'empty_or_non_string_input'
        return result

    try:
        msg = email.message_from_string(raw_text, policy=policy.compat32)

        result['message_id'] = msg.get('Message-ID')
        result['date']       = msg.get('Date')
        result['from']       = msg.get('From')
        result['to']         = msg.get('To')
        result['subject']    = msg.get('Subject')
        result['cc']         = msg.get('Cc') or msg.get('X-cc')
        result['bcc']        = msg.get('Bcc') or msg.get('X-bcc')
        result['x_from']     = msg.get('X-From')
        result['x_to']       = msg.get('X-To')
        result['x_folder']   = msg.get('X-Folder')
        result['x_origin']   = msg.get('X-Origin')
        result['x_filename'] = msg.get('X-FileName')
        result['content_type'] = msg.get('Content-Type')

        # Extract body, handling multipart messages
        body = ''
        if msg.is_multipart():
            parts = []
            for part in msg.walk():
                if part.get_content_type() == 'text/plain' and not part.is_multipart():
                    payload = part.get_payload(decode=True)
                    if payload:
                        charset = part.get_content_charset() or 'utf-8'
                        parts.append(payload.decode(charset, errors='replace'))
            body = '\n'.join(parts)
        else:
            payload = msg.get_payload(decode=True)
            if payload is not None:
                charset = msg.get_content_charset() or 'utf-8'
                body = payload.decode(charset, errors='replace')
            else:
                # get_payload(decode=True) can return None for malformed
                # Content-Transfer-Encoding; fall back to raw payload
                fallback = msg.get_payload()
                body = fallback if isinstance(fallback, str) else ''

        result['body'] = body.strip()
        result['parse_success'] = True

    except Exception as e:
        result['parse_error'] = f"{type(e).__name__}: {e}"

    return result


def parse_dataframe(df, message_col='message'):
    """Apply parse_email_message to every row and return a new dataframe
    with the original columns plus the extracted header/metadata columns.
    """
    parsed_records = df[message_col].apply(parse_email_message)
    parsed_df = pd.DataFrame(list(parsed_records))
    out = pd.concat([df.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)
    return out


In [ ]:
df_parsed = parse_dataframe(df_dedup, message_col='message')

print(f"Parsed {len(df_parsed):,} rows")
print(f"Successful parses: {df_parsed['parse_success'].sum():,}")
print(f"Failed parses:     {(~df_parsed['parse_success']).sum():,}")
df_parsed.head(3)


Parsed 517,401 rows
Successful parses: 517,401
Failed parses:     0


,file,message,message_id,date,from,to,subject,cc,bcc,x_from,x_to,x_folder,x_origin,x_filename,content_type,body,parse_success,parse_error
0,allen-p/_sent_mail/1.,"Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nConte...",<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,,,,Phillip K Allen,Tim Belden <Tim Belden/Enron@EnronXGate>,"\Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail",Allen-P,pallen (Non-Privileged).pst,text/plain; charset=us-ascii,Here is our forecast,True,None
1,allen-p/_sent_mail/10.,"Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May 2001 13:51:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: john.lavorato@enron.com\nSubject: Re:\nMime-Version: 1.0\n...",<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,,,Phillip K Allen,John J Lavorato <John J Lavorato/ENRON@enronXgate@ENRON>,"\Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail",Allen-P,pallen (Non-Privileged).pst,text/plain; charset=us-ascii,Traveling to have a business meeting takes the fun out of the trip. Especially if you have to prepare a presentation. I would suggest holding the business plan meetings here then take a trip wit...,True,None
2,allen-p/_sent_mail/100.,"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme>\nDate: Wed, 18 Oct 2000 03:00:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: leah.arsdall@enron.com\nSubject: Re: test\nMime-Version: ...",<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,,,Phillip K Allen,Leah Van Arsdall,\Phillip_Allen_Dec2000\Notes Folders\'sent mail,Allen-P,pallen.nsf,text/plain; charset=us-ascii,test successful. way to go!!!,True,None


In [ ]:
# Confirm the new metadata columns exist and were populated for successful parses
expected_cols = ['message_id', 'date', 'from', 'to', 'subject', 'x_from', 'x_to',
                 'x_folder', 'x_origin', 'x_filename', 'content_type', 'body',
                 'parse_success', 'parse_error']
for c in expected_cols:
    assert c in df_parsed.columns, f"Missing expected column: {c}"

success_rate = df_parsed['parse_success'].mean()
print(f"Parse success rate: {success_rate:.2%}")
assert success_rate > 0.95, "Unexpectedly low parse success rate — inspect df_parsed[~df_parsed.parse_success]"

# Spot check: message_id should be present for almost all successfully parsed rows
successful = df_parsed[df_parsed['parse_success']]
msg_id_rate = successful['message_id'].notna().mean()
print(f"Message-ID present rate (successful parses): {msg_id_rate:.2%}")

print()
print("Sample parsed record:")
sample = successful.iloc[0]
for c in ['message_id', 'date', 'from', 'to', 'subject', 'x_folder']:
    print(f"  {c:12s}: {sample[c]}")
print(f"  {'body':12s}: {str(sample['body'])[:120]}...")

print()
print("PASS: header/metadata extraction looks correct.")


Parse success rate: 100.00%
Message-ID present rate (successful parses): 100.00%

Sample parsed record:
  message_id  : <18782981.1075855378110.JavaMail.evans@thyme>
  date        : Mon, 14 May 2001 16:39:00 -0700 (PDT)
  from        : phillip.allen@enron.com
  to          : tim.belden@enron.com
  subject     : 
  x_folder    : \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
  body        : Here is our forecast...

PASS: header/metadata extraction looks correct.


In [ ]:
def flag_malformed(df):
    df = df.copy()

    reasons = []
    for _, row in df.iterrows():
        row_reasons = []
        if not row['parse_success']:
            row_reasons.append('parse_failed')
        else:
            if not row['from'] or str(row['from']).strip() == '':
                row_reasons.append('missing_from')
            if not row['date'] or str(row['date']).strip() == '':
                row_reasons.append('missing_date')
            if not row['body'] or str(row['body']).strip() == '':
                row_reasons.append('empty_body')
        reasons.append(row_reasons)

    df['malformed_reasons'] = reasons
    df['is_malformed'] = df['malformed_reasons'].apply(lambda r: len(r) > 0)
    return df


df_flagged = flag_malformed(df_parsed)

df_clean = df_flagged[~df_flagged['is_malformed']].reset_index(drop=True)
df_malformed = df_flagged[df_flagged['is_malformed']].reset_index(drop=True)

print(f"Clean rows:     {len(df_clean):,}")
print(f"Malformed rows: {len(df_malformed):,}  ({len(df_malformed)/len(df_flagged):.2%})")
print()
print("Breakdown of malformed reasons:")
from collections import Counter
reason_counts = Counter(r for reasons in df_malformed['malformed_reasons'] for r in reasons)
for reason, cnt in reason_counts.most_common():
    print(f"  {reason:16s}: {cnt:,}")


Clean rows:     517,401
Malformed rows: 0  (0.00%)

Breakdown of malformed reasons:


In [ ]:
# Every row in df_clean should have all critical fields populated
assert df_clean['parse_success'].all(), "df_clean contains a failed parse — filtering bug."
assert df_clean['from'].notna().all() and (df_clean['from'].str.strip() != '').all(), \
    "df_clean contains rows with missing 'from'."
assert df_clean['date'].notna().all() and (df_clean['date'].str.strip() != '').all(), \
    "df_clean contains rows with missing 'date'."
assert (df_clean['body'].str.strip() != '').all(), \
    "df_clean contains rows with an empty body."

# Every row in df_malformed should have at least one reason logged
assert df_malformed['malformed_reasons'].apply(len).gt(0).all(), \
    "Found a malformed row with no reason recorded — logic bug."

assert len(df_clean) + len(df_malformed) == len(df_flagged), \
    "Clean + malformed counts don't add up to the total — rows lost somewhere."

print("PASS: df_clean has no missing critical fields, all malformed rows have a logged reason.")


PASS: df_clean has no missing critical fields, all malformed rows have a logged reason.


In [ ]:
def content_hash(row):
    key = f"{row['from']}|{row['subject']}|{row['body']}"
    return hashlib.sha256(key.encode('utf-8', errors='replace')).hexdigest()

df_clean = df_clean.copy()
df_clean['content_hash'] = df_clean.apply(content_hash, axis=1)

n_before_final = len(df_clean)

# Dedup on message_id where it's present and non-blank
has_id = df_clean['message_id'].notna() & (df_clean['message_id'].str.strip() != '')
with_id = df_clean[has_id].drop_duplicates(subset=['message_id'], keep='first')
without_id = df_clean[~has_id]

df_stage1 = pd.concat([with_id, without_id]).sort_index()

# Dedup remaining rows on content hash (catches missing-Message-ID duplicates)
df_final = df_stage1.drop_duplicates(subset=['content_hash'], keep='first').reset_index(drop=True)

n_after_final = len(df_final)
print(f"Rows before final dedup: {n_before_final:,}")
print(f"Rows after final dedup:  {n_after_final:,}")
print(f"Removed as post-parse duplicates: {n_before_final - n_after_final:,}")


Rows before final dedup: 517,401
Rows after final dedup:  250,610
Removed as post-parse duplicates: 266,791


In [ ]:
dup_id = df_final[df_final['message_id'].notna() & (df_final['message_id'].str.strip() != '')] \
    .duplicated(subset=['message_id']).sum()
dup_hash = df_final.duplicated(subset=['content_hash']).sum()

assert dup_id == 0, f"Still {dup_id} duplicate Message-IDs remaining!"
assert dup_hash == 0, f"Still {dup_hash} duplicate content hashes remaining!"
assert len(df_final) <= n_before_final

print("PASS: no duplicate Message-IDs or content hashes remain in df_final.")
print(f"Final dataset size: {len(df_final):,} rows")


PASS: no duplicate Message-IDs or content hashes remain in df_final.
Final dataset size: 250,610 rows


In [ ]:
print("=" * 50)
print("PIPELINE SUMMARY")
print("=" * 50)
print(f"Raw rows loaded:              {len(df_raw):,}")
print(f"After exact-text dedup:       {len(df_dedup):,}")
print(f"After parsing:                {len(df_parsed):,}")
print(f"  - parse successes:          {df_parsed['parse_success'].sum():,}")
print(f"  - parse failures:           {(~df_parsed['parse_success']).sum():,}")
print(f"After malformed/missing drop: {len(df_clean) + len(df_malformed) - len(df_malformed):,} clean "
      f"/ {len(df_malformed):,} quarantined")
print(f"Final (post-parse dedup):     {len(df_final):,}")
print("=" * 50)

# Drop the helper column before saving if you don't need it downstream
output_cols = [c for c in df_final.columns if c != 'content_hash']
OUTPUT_PATH = "drive/MyDrive/phishing project datasets/enron_emails_cleaned.csv"

df_final[output_cols].to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset to: {OUTPUT_PATH}")

# Optional: also save the quarantined rows for inspection
MALFORMED_PATH = "drive/MyDrive/phishing project datasets/enron_emails_malformed.csv"
df_malformed.to_csv(MALFORMED_PATH, index=False)
print(f"Saved quarantined rows to: {MALFORMED_PATH}")


PIPELINE SUMMARY
Raw rows loaded:              517,401
After exact-text dedup:       517,401
After parsing:                517,401
  - parse successes:          517,401
  - parse failures:           0
After malformed/missing drop: 517,401 clean / 0 quarantined
Final (post-parse dedup):     250,610
Saved cleaned dataset to: drive/MyDrive/phishing project datasets/enron_emails_cleaned.csv
Saved quarantined rows to: drive/MyDrive/phishing project datasets/enron_emails_malformed.csv


In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250610 entries, 0 to 250609
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   file               250610 non-null  object
 1   message            250610 non-null  object
 2   message_id         250610 non-null  object
 3   date               250610 non-null  object
 4   from               250610 non-null  object
 5   to                 241182 non-null  object
 6   subject            250610 non-null  object
 7   cc                 250581 non-null  object
 8   bcc                250581 non-null  object
 9   x_from             250581 non-null  object
 10  x_to               250581 non-null  object
 11  x_folder           250581 non-null  object
 12  x_origin           250581 non-null  object
 13  x_filename         250581 non-null  object
 14  content_type       250581 non-null  object
 15  body               250610 non-null  object
 16  parse_success      2